# Vivado TB Artifact Generator — MultiPulse Demo

Generates all files needed to drive `tb_qick_emu.sv` in Vivado:
- `pmem.mem` — tproc program memory
- `wmem.mem` — tproc wave parameter table
- `sgmem_ch{N}.mem` — signal generator envelope tables (for shaped pulses)
- `axi_replay.txt` — AXI-Lite register write replay log

**No board required.** This notebook runs entirely in Python against the emulated SoC.

After running, copy the output directory to the Vivado machine and point `EMU_DIR` in `tb_qick_emu.sv` at it.

In [1]:
import sys
import importlib
import pathlib
import numpy as np
import matplotlib.pyplot as plt

# Path setup
REPO_ROOT  = pathlib.Path.cwd().parent          # emulator/
source_dir = REPO_ROOT / 'software' / 'source'
if str(source_dir) not in sys.path:
    sys.path.insert(0, str(source_dir))

import qick_emu
importlib.reload(qick_emu)
from qick_emu import QickEmu
from qick.asm_v2 import AveragerProgramV2

print(f"REPO_ROOT : {REPO_ROOT}")
print(f"source_dir: {source_dir}")

REPO_ROOT : /Users/sbf/Desktop/to_verilate/emulator
source_dir: /Users/sbf/Desktop/to_verilate/emulator/software/source


/Users/sbf/projects/qick-env/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Board config + emulator setup
CFG_PATH = REPO_ROOT / 'notebooks' / 'qick_config_216.json'
emu      = QickEmu(str(CFG_PATH))
soccfg   = emu.soccfg

# Output directory for Vivado artifacts
# Copy this folder to the Vivado machine and set EMU_DIR to its path.
ARTIFACTS_DIR = REPO_ROOT / 'artifacts' / 'multi_pulse'
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Artifacts will be written to: {ARTIFACTS_DIR}")

# Program constants
GEN_CH    = 0
RO_CH     = 0
FREQ      = 100.0   # MHz
TRIG_TIME = 0.0     # us

QICK library version mismatch: 0.2.366 remote (the board), 0.2.370 local (the PC)
                        This may cause errors, usually KeyError in QickConfig initialization.
                        If this happens, you must bring your versions in sync.


Artifacts will be written to: /Users/sbf/Desktop/to_verilate/emulator/artifacts/multi_pulse


In [3]:
class MultiPulseProgram(AveragerProgramV2):
    def _initialize(self, cfg):
        ro_ch  = cfg['ro_ch']
        gen_ch = cfg['gen_ch']
        self.declare_gen(ch=gen_ch, nqz=1)
        self.declare_readout(ch=ro_ch, length=cfg['ro_len'])
        ramp_len = 0.2
        self.add_gauss(ch=gen_ch, name="ramp", sigma=ramp_len/10,
                       length=ramp_len, even_length=True)
        self.add_pulse(ch=gen_ch, name="myflattop", ro_ch=ro_ch,
                       style="flat_top", envelope="ramp",
                       freq=cfg['freq'], length=0.1, phase=0, gain=1.0)
        self.add_pulse(ch=gen_ch, name="mygaus", ro_ch=ro_ch,
                       style="arb", envelope="ramp",
                       freq=cfg['freq'], phase=0, gain=1.0)
        self.add_pulse(ch=gen_ch, name="myconst", ro_ch=ro_ch,
                       style="const", length=0.2,
                       freq=cfg['freq'], phase=0, gain=1.0)
        self.add_pulse(ch=gen_ch, name="myflattop2", ro_ch=ro_ch,
                       style="flat_top", envelope="ramp",
                       freq=cfg['freq'], length=0.1, phase=90, gain=1.0)
        self.add_readoutconfig(ch=ro_ch, name="myro",
                       freq=cfg['freq'], gen_ch=gen_ch, length=cfg['ro_len'])
        self.send_readoutconfig(ch=ro_ch, name="myro", t=0)
    def _body(self, cfg):
        self.trigger(ros=[cfg['ro_ch']], pins=[0],
                     t=cfg['trig_time'], ddr4=True)
        self.pulse(ch=cfg['gen_ch'], name="myflattop",  t=0.0)
        self.pulse(ch=cfg['gen_ch'], name="mygaus",     t=0.4)
        self.pulse(ch=cfg['gen_ch'], name="myconst",    t=0.8)
        self.pulse(ch=cfg['gen_ch'], name="myflattop2", t=1.2)
        self.pulse(ch=cfg['gen_ch'], name="mygaus",     t=1.6)
config = {
    'gen_ch':    GEN_CH,
    'ro_ch':     RO_CH,
    'freq':      FREQ,
    'trig_time': TRIG_TIME,
    'ro_len':    3,
}

prog = MultiPulseProgram(soccfg, reps=1, final_delay=0.5, cfg=config)
print("Program created.")

Program created.


In [ ]:
# Generate all emulator artifacts
soc  = emu.make_soc(memdir=ARTIFACTS_DIR)
prep = emu.prepare(prog, soc, memdir=ARTIFACTS_DIR)

# Convert axi_replay.jsonl -> axi_replay.txt for tb_qick_emu.sv
# Also prints the localparam values to paste into tb_qick_emu.sv
emu.export_vivado_files(memdir=ARTIFACTS_DIR)

print(f"\nArtifacts written to: {ARTIFACTS_DIR}")
print("\nFiles generated:")
for f in sorted(ARTIFACTS_DIR.iterdir()):
    size = f.stat().st_size
    print(f"  {f.name:<25s}  {size:>8d} bytes")

In [ ]:
# Inspect assembled program
print("=== Program memory (pmem) ===")
prog.print_pmem2hex()

In [ ]:
# Inspect AXI replay transactions
print("=== AXI replay (axi_replay.txt) ===")
replay_txt = ARTIFACTS_DIR / 'axi_replay.txt'
with replay_txt.open() as f:
    for line in f:
        print(line, end='')

In [ ]:
import json
cfg = json.load(open('emulator/notebooks/qick_config_216.json'))
for i, g in enumerate(cfg['gens']):
    print(f"gen[{i}]: type={g['type']}, tproc_ch={g.get('tproc_ch', 'N/A')}, fullpath={g['fullpath']}"

In [ ]:
# Inspect signal generator envelope (sgmem_ch0.mem)
# This program uses flat_top and arb pulses so envelope data IS expected.
sgmem = ARTIFACTS_DIR / 'sgmem_ch0.mem'
if sgmem.exists():
    lines = sgmem.read_text().strip().splitlines()
    print(f"sgmem_ch0.mem: {len(lines)} I,Q sample pairs")
    IQ = np.array([[int(v) for v in l.split(',')] for l in lines])
    plt.figure(figsize=(10, 3))
    plt.plot(IQ[:, 0], label='I')
    plt.plot(IQ[:, 1], label='Q')
    plt.title('SG ch0 envelope table (sgmem_ch0.mem)')
    plt.xlabel('sample index')
    plt.ylabel('amplitude (int16)')
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("sgmem_ch0.mem not generated (const-only pulses have no envelope)")

## Vivado Setup Instructions

1. **Copy** the `artifacts/multi_pulse/` folder to your Vivado machine.

2. **Open** `firmware/testbench/qick_testbench/qick_testbench.xpr` in Vivado.

3. **Add** `firmware/testbench/qick_testbench/src/tb/tb_qick_emu.sv` as a simulation source.

4. **Set top module** to `tb_qick_emu` for the `sim_1` fileset.

5. **Edit `EMU_DIR`** in `tb_qick_emu.sv` to point at the copied folder.
   From the Vivado xsim working directory (`*.sim/sim_1/behav/xsim/`), a relative path like:
   ```
   string EMU_DIR = "../../../../src/tb/multi_pulse";
   ```
   works if you place the files under `src/tb/multi_pulse/`.  
   Or use an absolute path.

6. **Paste the localparam values** printed by `export_vivado_files()` above into `tb_qick_emu.sv`
   (only needed if they differ from the defaults — the defaults match ZCU216).

7. **Run Behavioral Simulation** — CSV outputs appear in `EMU_DIR`:
   - `dac_out.csv` — DAC samples (16 parallel, one row per SG clock)
   - `avg_out.csv` — averaged IQ from avg_buffer
   - `dec_out.csv` — decimated IQ from avg_buffer

### Pulse sequence in this program
```
t=0.0 us  myflattop   flat_top  (gauss envelope, gain=1.0, phase=0°)
t=0.4 us  mygaus      arb       (gauss envelope, gain=1.0, phase=0°)
t=0.8 us  myconst     const     (no envelope,    gain=1.0, phase=0°)
t=1.2 us  myflattop2  flat_top  (gauss envelope, gain=1.0, phase=90°)
t=1.6 us  mygaus      arb       (gauss envelope, gain=1.0, phase=0°)
```
Readout trigger fires at `t=TRIG_TIME`, window length = 1.9 µs.

**Expected `sgmem_ch0.mem`**: present (flat_top and arb pulses use the gauss envelope).  
**Expected `avg_out.csv`/`dec_out.csv`**: populated only if `TEST_RUN_TIME` in the SV is long enough (~3 µs minimum).